# TTFT Evaluator (Time-To-First-Token)

Measures **Time-To-First-Token (TTFT)** — the elapsed time from sending a request until the first token arrives in a streaming response.

TTFT is what the user *feels*: even if total generation takes 20 seconds, a 0.3s TTFT makes the experience feel snappy. It grows with context length because the model must encode the full prompt before generating the first token.

The evaluator is **data-agnostic**: it accepts any `Dict[str, str]` mapping a label to a context string.

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelTTFTEvaluator
from bellmira.utils.context_utils import (
    contexts_from_word_counts,
    contexts_from_files,
    contexts_from_bible,
    read_text_file,
)

## 2. Build contexts

Pick the approach that fits your data source.

### Option A — word-count truncation (any text)

In [ ]:
MY_TEXT_PATH = "/workspaces/BeLLMira/resources/data/text/bible.txt"  # replace with any .txt
text = read_text_file(MY_TEXT_PATH)

# Sweep from ~500 words (~750 tokens) to ~8000 words (~12k tokens)
contexts = contexts_from_word_counts(
    text,
    word_counts=[500, 1000, 2000, 4000, 8000],
)
print({k: f"{len(v.split())} words" for k, v in contexts.items()})

### Option B — one file per context

In [ ]:
# contexts = contexts_from_files([
#     "/path/to/doc_short.txt",
#     "/path/to/doc_medium.txt",
#     "/path/to/doc_long.txt",
# ])

### Option C — Bible corpus (bundled)

In [ ]:
# contexts = contexts_from_bible(
#     bible_path="/workspaces/BeLLMira/resources/data/text/bible.txt",
#     chapter_numbers=[2, 5, 9, 14, 22, 35],
# )

## 3. Configuration

In [ ]:
MODEL_URL = "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/"

PROMPTS = [
    "What is the main theme of the text above?",
    "Summarise the key events described in this passage.",
]

## 4. Initialise the evaluator

In [ ]:
evaluator = ModelTTFTEvaluator(
    url=MODEL_URL,
    prompts=PROMPTS,
    contexts=contexts,
    temperature=0.0,
)

## 5. Run evaluation

In [ ]:
results = evaluator.evaluate(max_prompts=1)
results

## 6. Inspect TTFT vs context size

In [ ]:
averages = evaluator.compute_averages(results)

print(f"{'Context':<20}  {'Avg TTFT':>10}  {'Avg Total':>10}  {'Avg Chars':>10}")
print("-" * 55)
for ctx_key, avg in averages.items():
    print(f"{ctx_key:<20}  {str(avg.get('Avg_TTFT')) + 's':>10}  "
          f"{str(avg.get('Avg_total_time')) + 's':>10}  "
          f"{str(avg.get('Avg_output_chars')):>10}")

## 7. Extract flat metrics for logging

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)
for key, val in metrics.items():
    print(f"{key}: {val}")

## 8. Compare TTFT across two models

In [ ]:
models = {
    "Qwen3-4B":  "https://chatqwen3-4-backend.dat-aip-millm.qa.mbcp.cloud/",
    "Qwen3-14B": "https://chatqwen3-14-backend.dat-aip-millm.qa.mbcp.cloud/",
}

# Build a smaller context set for the comparison sweep
comparison_contexts = contexts_from_word_counts(text, word_counts=[500, 2000, 6000])

all_metrics = {}
for model_name, url in models.items():
    ev = ModelTTFTEvaluator(
        url=url,
        prompts=PROMPTS[:1],
        contexts=comparison_contexts,
    )
    res = ev.evaluate(max_prompts=1)
    all_metrics[model_name] = ev.extract_threshold_metrics(res)

import pprint
pprint.pprint(all_metrics)